# TfL API — Exploratory Data Analysis

## Import packages

In [15]:
import polars as pl
import requests
import os
import boto3
import json
import requests
from dotenv import load_dotenv
import gzip, json
from datetime import datetime, timezone

## Get Modes

In [12]:
call_modes_api = requests.get("https://api.tfl.gov.uk/Line/Meta/Modes")
modes_json = call_modes_api.json()

print(f"Modes JSON type: {type(modes_json)}")
print(f"Modes JSON length: {len(modes_json)}")
# create a list of tuples with key and type for each key in the first item of the modes_json list
print(
    f"Mode types Keys and Types: {[(key, type(value)) for key, value in (modes_json[0].items())]}"
)
print(f"Modes JSON first item: {modes_json[0]}")

df_modes = pl.DataFrame(modes_json)

# show full list of modes in the df_modes dataframe, without concatenation
# write to temp output file
df_modes.select("modeName").unique("modeName").sort("modeName").write_csv("example_data/modes.csv")

Modes JSON type: <class 'list'>
Modes JSON length: 18
Mode types Keys and Types: [('$type', <class 'str'>), ('isTflService', <class 'bool'>), ('isFarePaying', <class 'bool'>), ('isScheduledService', <class 'bool'>), ('modeName', <class 'str'>)]
Modes JSON first item: {'$type': 'Tfl.Api.Presentation.Entities.Mode, Tfl.Api.Presentation.Entities', 'isTflService': True, 'isFarePaying': True, 'isScheduledService': True, 'modeName': 'bus'}


## Get Lines

In [13]:
lines_api_call = requests.get("https://api.tfl.gov.uk/Line/Mode/tube")

lines_json = lines_api_call.json()

print(f"Lines JSON type: {type(lines_json)}")
print(f"Line status JSON length: {len(lines_json)}")
# create a list of tuples with key and type for each key in the first item of the lines_json list
print(f"Line types Keys and Types: {[(key, type(value)) for key, value in (lines_json[0].items())]}")
print(f"Line status JSON first item: {lines_json[0]}")

# Convert to polars df
lines_df = pl.DataFrame(lines_json)

lines_df.head()

Lines JSON type: <class 'list'>
Line status JSON length: 11
Line types Keys and Types: [('$type', <class 'str'>), ('id', <class 'str'>), ('name', <class 'str'>), ('modeName', <class 'str'>), ('disruptions', <class 'list'>), ('created', <class 'str'>), ('modified', <class 'str'>), ('lineStatuses', <class 'list'>), ('routeSections', <class 'list'>), ('serviceTypes', <class 'list'>), ('crowding', <class 'dict'>)]
Line status JSON first item: {'$type': 'Tfl.Api.Presentation.Entities.Line, Tfl.Api.Presentation.Entities', 'id': 'bakerloo', 'name': 'Bakerloo', 'modeName': 'tube', 'disruptions': [], 'created': '2026-09-15T13:26:22.117Z', 'modified': '2026-09-15T13:26:22.117Z', 'lineStatuses': [], 'routeSections': [], 'serviceTypes': [{'$type': 'Tfl.Api.Presentation.Entities.LineServiceTypeInfo, Tfl.Api.Presentation.Entities', 'name': 'Regular', 'uri': '/Line/Route?ids=Bakerloo&serviceTypes=Regular'}], 'crowding': {'$type': 'Tfl.Api.Presentation.Entities.Crowding, Tfl.Api.Presentation.Entities'

$type,id,name,modeName,disruptions,created,modified,lineStatuses,routeSections,serviceTypes,crowding
str,str,str,str,list[null],str,str,list[null],list[null],list[struct[3]],struct[1]
"""Tfl.Api.Presentation.Entities.…","""bakerloo""","""Bakerloo""","""tube""",[],"""2026-09-15T13:26:22.117Z""","""2026-09-15T13:26:22.117Z""",[],[],"[{""Tfl.Api.Presentation.Entities.LineServiceTypeInfo, Tfl.Api.Presentation.Entities"",""Regular"",""/Line/Route?ids=Bakerloo&serviceTypes=Regular""}]","{""Tfl.Api.Presentation.Entities.Crowding, Tfl.Api.Presentation.Entities""}"
"""Tfl.Api.Presentation.Entities.…","""central""","""Central""","""tube""",[],"""2026-09-15T13:26:22.117Z""","""2026-09-15T13:26:22.117Z""",[],[],"[{""Tfl.Api.Presentation.Entities.LineServiceTypeInfo, Tfl.Api.Presentation.Entities"",""Regular"",""/Line/Route?ids=Central&serviceTypes=Regular""}, {""Tfl.Api.Presentation.Entities.LineServiceTypeInfo, Tfl.Api.Presentation.Entities"",""Night"",""/Line/Route?ids=Central&serviceTypes=Night""}]","{""Tfl.Api.Presentation.Entities.Crowding, Tfl.Api.Presentation.Entities""}"
"""Tfl.Api.Presentation.Entities.…","""circle""","""Circle""","""tube""",[],"""2026-09-15T13:26:22.117Z""","""2026-09-15T13:26:22.117Z""",[],[],"[{""Tfl.Api.Presentation.Entities.LineServiceTypeInfo, Tfl.Api.Presentation.Entities"",""Regular"",""/Line/Route?ids=Circle&serviceTypes=Regular""}]","{""Tfl.Api.Presentation.Entities.Crowding, Tfl.Api.Presentation.Entities""}"
"""Tfl.Api.Presentation.Entities.…","""district""","""District""","""tube""",[],"""2026-09-15T13:26:22.117Z""","""2026-09-15T13:26:22.117Z""",[],[],"[{""Tfl.Api.Presentation.Entities.LineServiceTypeInfo, Tfl.Api.Presentation.Entities"",""Regular"",""/Line/Route?ids=District&serviceTypes=Regular""}]","{""Tfl.Api.Presentation.Entities.Crowding, Tfl.Api.Presentation.Entities""}"
"""Tfl.Api.Presentation.Entities.…","""hammersmith-city""","""Hammersmith & City""","""tube""",[],"""2026-09-15T13:26:22.117Z""","""2026-09-15T13:26:22.117Z""",[],[],"[{""Tfl.Api.Presentation.Entities.LineServiceTypeInfo, Tfl.Api.Presentation.Entities"",""Regular"",""/Line/Route?ids=Hammersmith & City&serviceTypes=Regular""}]","{""Tfl.Api.Presentation.Entities.Crowding, Tfl.Api.Presentation.Entities""}"


## Get StopPoints

In [14]:
# Test calling TFL API with token
# List of stop points available at mode "tube"
stop_points_api_call = requests.get(
    "https://api.tfl.gov.uk/StopPoint/Type/NaptanMetroStation"
)
# ?app_id=YOUR_APP_ID&app_key=YOUR_APP_KEY"

# convert to json object
stop_point_json = stop_points_api_call.json()

# print(stop_point_json[0])  # Print the first stop point in the list

# care about naptanId, commonName, lat, lon, modes, stopType - not sure about the others

# look at all the keys in the first stop point
print(f"Stop point JSON type: {type(stop_point_json)}")
print(f"Stop point JSON length: {len(stop_point_json) if isinstance(stop_point_json, list) else list(stop_point_json.keys())}")
# Keys and key types together as key value pairs
print(f"Stop point JSON keys and types: {[(key, type(value)) for key, value in (stop_point_json[0].items() if isinstance(stop_point_json, list) else stop_point_json.items())]}")
print(f"Stop point JSON first item: {stop_point_json[0] if isinstance(stop_point_json, list) else stop_point_json}")


# filter to tube
tube_stop_points = [stop_point for stop_point in stop_point_json if "tube" in stop_point.get("modes", [])]
print(tube_stop_points[0])  # Print the first tube stop point in the list
# select certain fields from the tube stop points
tube_stop_points = [{key: stop_point.get(key) for key in ["naptanId", "commonName", "modes"]} for stop_point in tube_stop_points]
print(tube_stop_points[0])  # Print the first tube stop point in the list with selected fields

# Total number of tube stop points
print(f"Total number of tube stop points: {len(tube_stop_points)}") # Same as what it says online 272

Stop point JSON type: <class 'list'>
Stop point JSON length: 949
Stop point JSON keys and types: [('$type', <class 'str'>), ('naptanId', <class 'str'>), ('modes', <class 'list'>), ('stopType', <class 'str'>), ('lines', <class 'list'>), ('lineGroup', <class 'list'>), ('lineModeGroups', <class 'list'>), ('status', <class 'bool'>), ('id', <class 'str'>), ('commonName', <class 'str'>), ('placeType', <class 'str'>), ('additionalProperties', <class 'list'>), ('children', <class 'list'>), ('lat', <class 'float'>), ('lon', <class 'float'>)]
Stop point JSON first item: {'$type': 'Tfl.Api.Presentation.Entities.StopPoint, Tfl.Api.Presentation.Entities', 'naptanId': '490G00002264', 'modes': ['bus'], 'stopType': 'NaptanMetroStation', 'lines': [], 'lineGroup': [], 'lineModeGroups': [], 'status': True, 'id': '490G00002264', 'commonName': 'Chauntler Close / Cundy Park', 'placeType': 'StopPoint', 'additionalProperties': [], 'children': [], 'lat': 0.0, 'lon': 0.0}
{'$type': 'Tfl.Api.Presentation.Entitie

## Get Line Disruptions

In [15]:
# This is the one I want
line_status_api_call = requests.get(
    "https://api.tfl.gov.uk/Line/Mode/tube/Status?detail=true"
)

line_status_json = line_status_api_call.json()

print(f"Line status JSON type: {type(line_status_json)}")
print(
    f"Line status JSON length: {len(line_status_json) if isinstance(line_status_json, list) else list(line_status_json.keys())}"
)
# Keys and key types together as key value pairs
print(
    f"Line status JSON keys and types: {[(key, type(value)) for key, value in (line_status_json[0].items() if isinstance(line_status_json, list) else line_status_json.items())]}"
)
print(
    f"Line status JSON first item: {line_status_json[0] if isinstance(line_status_json, list) else line_status_json}"
)

Line status JSON type: <class 'list'>
Line status JSON length: 11
Line status JSON keys and types: [('$type', <class 'str'>), ('id', <class 'str'>), ('name', <class 'str'>), ('modeName', <class 'str'>), ('disruptions', <class 'list'>), ('created', <class 'str'>), ('modified', <class 'str'>), ('lineStatuses', <class 'list'>), ('routeSections', <class 'list'>), ('serviceTypes', <class 'list'>), ('crowding', <class 'dict'>)]
Line status JSON first item: {'$type': 'Tfl.Api.Presentation.Entities.Line, Tfl.Api.Presentation.Entities', 'id': 'bakerloo', 'name': 'Bakerloo', 'modeName': 'tube', 'disruptions': [], 'created': '2026-09-15T13:26:22.117Z', 'modified': '2026-09-15T13:26:22.117Z', 'lineStatuses': [{'$type': 'Tfl.Api.Presentation.Entities.LineStatus, Tfl.Api.Presentation.Entities', 'id': 0, 'statusSeverity': 10, 'statusSeverityDescription': 'Good Service', 'created': '0001-01-01T00:00:00', 'validityPeriods': []}], 'routeSections': [], 'serviceTypes': [{'$type': 'Tfl.Api.Presentation.Ent

## Get Disruptions at Stop Points

In [16]:
# This get things like lift outages, step free access, etc. for the tube network
current_disruptions_api_call = requests.get(
    "https://api.tfl.gov.uk/StopPoint/Mode/tube/Disruption?includeRouteBlockedStops=true"
)

current_disruptions_json = current_disruptions_api_call.json()

# Don't really want this

# Write to S3

In [ ]:
import os
import boto3
import json
import requests
from dotenv import load_dotenv
import gzip, json
from datetime import datetime, timezone

# Load environment variables from .env file
load_dotenv("../.env")

bucket = os.environ.get("S3_BUCKET")
prefix = os.environ.get("S3_PREFIX", "")

if not bucket:
    raise RuntimeError("S3_BUCKET is not set")

print("Downloading API data from TFL...")
api_response = requests.get("https://api.tfl.gov.uk/Line/Meta/Modes")
api_response.raise_for_status()
api_response_dict = api_response.json()
print("Downloaded API data from TFL, now uploading to S3...")


# gzip the JSON data before uploading to S3
polled_at = datetime.now(timezone.utc)
# Create a record with the polled_at timestamp and the API response
record = {"polled_at": polled_at.isoformat(), "response": api_response_dict}
# json.dumps() converts the Python dict into a JSON string
# + "\n" ends the record with a newline. Strictly, with one poll per file, Athena would most likely read the file without it. It's there for when files get combined
line = json.dumps(record) + "\n"
# gzip.compress works on bytes, so you need a string first, which .encode("utf-8") then turns into bytes
body = gzip.compress(line.encode("utf-8"))


folder = "modes" # This will change for each api get
key = (
    f"raw/tfl/{folder}/poll_date={polled_at:%Y-%m-%d}/{polled_at:%Y%m%dT%H%M%SZ}.json.gz"
)

# Write to local file for debugging
# with open("example_data/modes.json", "w", encoding="utf-8") as output_file:
#  json.dump(api_response_dict, output_file)
#  output_file.write("\n")


# Use environment credentials or the notebook's IAM role instead of
# a missing local "default" profile.
if os.environ.get("AWS_PROFILE") == "default":
    os.environ.pop("AWS_PROFILE")

print(f"Uploading to S3 bucket: {bucket}, key: {key}")
s3_client = boto3.client("s3")

s3_client.put_object(
    Bucket=bucket,
    Key=key,
    Body=json.dumps(api_response_dict).encode("utf-8"),
    ContentType="application/json",
)
print(f"Uploaded API data to S3 bucket: {bucket}, key: {key}")

## Functions to write to S3

In [16]:
from datetime import datetime, timezone

# Download data from api - need to enhance with try catch block and logging, and also to handle the case where the API is down or returns an error

def call_api(url:str):
    """
    Call the TFL API and return the response as a JSON object.
    """
    print(f"Downloading API data from TFL: {url}")
    api_response = requests.get(url)
    api_response.raise_for_status()
    return api_response.json()

def poll_datetime():
    """
    Return the current UTC datetime for the polled_at timestamp.
    """
    return datetime.now(timezone.utc)

def transform_data(api_response_dict: dict, polled_at: datetime):
    """
    Transform the API response into a list of records with the polled_at timestamp and then gzip the JSON data
    """
    print("Transforming API data from TFL...")
    # Create a record with the polled_at timestamp and the API response
    record = {"polled_at": polled_at.isoformat(), "response": api_response_dict}
    # json.dumps() converts the Python dict into a JSON string
    # + "\n" ends the record with a newline. Strictly, with one poll per file, Athena would most likely read the file without it. It's there for when files get combined
    line = json.dumps(record) + "\n"
    # gzip.compress works on bytes, so you need a string first, which .encode("utf-8") then turns into bytes
    body = gzip.compress(line.encode("utf-8"))
    return body

def generate_s3_key(folder: str, polled_at: datetime):
    """
    Generate the S3 key for the uploaded file based on the folder and polled_at timestamp.
    """
    return f"raw/tfl/{folder}/poll_date={polled_at:%Y-%m-%d}/{polled_at:%Y%m%dT%H%M%SZ}.json.gz"

def upload_to_s3(bucket: str, key: str, body: bytes):
    """
    Upload the gzipped JSON data to S3.
    """
    print(f"Uploading to S3 bucket: {bucket}, key: {key}")
    s3_client = boto3.client("s3")
    s3_client.put_object(
        Bucket=bucket,
        Key=key,
        Body=body,
        ContentType="application/json",
    )
    print(f"Uploaded API data to S3 bucket: {bucket}, key: {key}")




In [17]:
# Define accepted modes for the TFL API
accepted_modes = ["dlr", "elizabeth-line", "overground", "tube"]

# Take the staging list, return final list
def get_api_urls(stg_api_dict: list[dict], final_api_dict: list[dict]) -> None:
    """Generate API URLs for each accepted mode."""
    for record in stg_api_dict:
        base_url = record["url"]
        modes = record["accepted_modes"]
        folder = record["folder"]
        frequency = record["frequency"]

        if modes:
            for mode in modes:
                final_api_dict.append(
                    {   
                        "url": base_url.format(modes=mode),
                        "folder": folder,
                        "frequency": frequency,
                    }
                )
        else:
            final_api_dict.append(
                {
                    "url": base_url,
                    "folder": folder,
                    "frequency": frequency,
                }
            )


# Initial API calls to get the modes and lines for each mode
stg_api_dict = [
{"url": "https://api.tfl.gov.uk/Line/Meta/Modes", "accepted_modes": [], "folder": "modes", "frequency": "weekly"},
{"url": "https://api.tfl.gov.uk/Line/Mode/{modes}", "accepted_modes": accepted_modes, "folder": "lines", "frequency": "weekly"},
{"url": "https://api.tfl.gov.uk/StopPoint/Type/{modes}", "accepted_modes": accepted_modes, "folder": "stop_points", "frequency": "weekly"},
{"url": "https://api.tfl.gov.uk/Line/Meta/Severity", "accepted_modes": [], "folder": "severity", "frequency": "weekly"},
{"url": "https://api.tfl.gov.uk/Line/Meta/DisruptionCategories", "accepted_modes": [], "folder": "disruption_categories", "frequency": "weekly"},
{"url": "https://api.tfl.gov.uk/Line/Mode/{modes}/Status", "accepted_modes": accepted_modes, "folder": "line_status", "frequency": "twice_daily"},
{"url": "https://api.tfl.gov.uk/StopPoint/Mode/{modes}/Disruption", "accepted_modes": accepted_modes, "folder": "stop_point_disruptions", "frequency": "weekly"}
]

# Initialise final api calls list
final_api_dict = []


get_api_urls(stg_api_dict, final_api_dict)

print(len(final_api_dict))

# Test just the first api in my dict

test_api = final_api_dict[0]
print(f"Test API call: {test_api}")

# Load environment variables from .env file
load_dotenv("../.env")

bucket = os.environ.get("S3_BUCKET")
prefix = os.environ.get("S3_PREFIX", "")

polled_at = poll_datetime()
key = generate_s3_key(test_api["folder"], polled_at)
response_dict = call_api(test_api["url"])
body = transform_data(response_dict, polled_at)
upload_to_s3(bucket, key, body)

19
Test API call: {'url': 'https://api.tfl.gov.uk/Line/Meta/Modes', 'folder': 'modes', 'frequency': 'weekly'}
Transforming API data from TFL...
Uploading to S3 bucket: tb-demo-data-428792078678-eu-west-2-an, key: raw/tfl/modes/poll_date=2026-09-21/20260921T212458Z.json.gz
Uploaded API data to S3 bucket: tb-demo-data-428792078678-eu-west-2-an, key: raw/tfl/modes/poll_date=2026-09-21/20260921T212458Z.json.gz
